# 8장. 표 자료에서의 신경망

범주를 숫자로 바꾸는 방법을 정하고, 1권의 그래디언트 부스팅과 같은 자료에서 다시 비교한다.

**준비:** 국토교통부 실거래가 공개시스템에서 받은 서울 아파트 매매 CSV 두 개(2023년, 2024년 계약분)를 왼쪽 파일 창에 올린 뒤 위에서부터 차례로 실행한다. 끝까지 실행하는 데 10분쯤 걸린다.

## 8.1 범주의 번호를 숫자로 넣기와 원-핫 인코딩

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
from torch import nn

torch.set_num_threads(1)
import os, re

def period(path):
    """파일 머리말에서 계약일자 범위를 읽는다. 실거래가 파일이 아니면 빈 리스트를 돌려준다."""
    with open(path, encoding="cp949", errors="ignore") as f:
        head = "".join(f.readline() for _ in range(15))
    return re.findall(r"계약일자 : (\d{4}-\d{2}-\d{2}) ~ (\d{4}-\d{2}-\d{2})", head)

files = sorted(f for f in os.listdir(".") if f.endswith(".csv") and period(f))
raw = pd.concat([pd.read_csv(f, encoding="cp949", skiprows=15, thousands=",",
                             dtype={"번지": str, "본번": str, "부번": str, "동": str}) for f in files],
                ignore_index=True)

v = raw[(raw["해제사유발생일"] == "-") & (raw["거래유형"] != "직거래")].copy()
v = v.rename(columns={"전용면적(㎡)": "면적"})
v["억"] = v["거래금액(만원)"] / 1e4
v["날짜"] = pd.to_datetime(v["계약년월"].astype(str) + v["계약일"].astype(str).str.zfill(2), format="%Y%m%d")
v["구"] = v["시군구"].str.split().str[1]
v["동네"] = v["시군구"].str.split().str[2]
v["동이름"] = v["구"] + " " + v["동네"]
v["단지"] = v["시군구"] + " " + v["번지"] + " " + v["단지명"]
v["연"] = v["날짜"].dt.year
v["연식"] = v["연"] - v["건축년도"]
v["달"] = (v["연"] - 2023) * 12 + v["날짜"].dt.month          # 2023년 1월이 1, 2025년 12월이 36
v = v.sort_values("날짜").reset_index(drop=True)
print(len(files), raw.shape, len(v))

In [ ]:
from scipy import sparse
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor

def tables(ref):
    """ref의 동과 단지에 번호를 매기고, 트리용 표와 릿지용 표를 만드는 함수를 돌려준다."""
    dongs = pd.Index(sorted(ref["동이름"].unique()))
    complexes = pd.Index(sorted(ref["단지"].unique()))
    def for_trees(df):
        dong_no = dongs.get_indexer(df["동이름"])
        complex_no = complexes.get_indexer(df["단지"])
        plain = [df["면적"], df["층"], df["건축년도"]]
        return np.column_stack([*plain, dong_no, complex_no, df["달"]])
    def for_line(df):
        age = (2023 + (df["달"] - 1) // 12) - df["건축년도"]
        decades = [(np.clip(age, 0, 59) // 10 == k).astype(float) for k in range(1, 6)]
        num = np.column_stack([np.log(df["면적"]), df["층"], *decades, df["달"]])
        blocks = [sparse.csr_matrix(num)]
        for col, cats in [("동이름", dongs), ("단지", complexes)]:
            where = cats.get_indexer(df[col])
            known = where >= 0
            ones = (np.ones(known.sum()), (np.where(known)[0], where[known]))
            blocks.append(sparse.csr_matrix(ones, shape=(len(df), len(cats))))
        return sparse.hstack(blocks).tocsr()
    return for_trees, for_line

def price(df):
    return np.log(df["억"].to_numpy())

def fit(made, kind, lam=0.01, rounds=1000, seed=0):
    """made로 모델을 추정하고, 거래 표를 받아 로그 가격을 예측하는 함수를 돌려준다."""
    for_trees, for_line = tables(made)
    line = Ridge(alpha=lam, solver="sparse_cg", tol=1e-8, max_iter=20000)
    line.fit(for_line(made), price(made))
    if kind == "릿지":
        return lambda df: line.predict(for_line(df))
    mend = HistGradientBoostingRegressor(learning_rate=0.1, max_iter=rounds,
                                         early_stopping=False, random_state=seed)
    # 릿지의 잔차를 부스팅이 예측한다
    mend.fit(for_trees(made), price(made) - line.predict(for_line(made)))
    return lambda df: line.predict(for_line(df)) + mend.predict(for_trees(df))

def off(actual_log, predicted_log):
    """오차율의 중앙값(%)."""
    ratio = np.abs(np.exp(predicted_log - actual_log) - 1)
    return round(float(np.median(ratio) * 100), 2)

In [ ]:
a23, a24 = v[v["연"] == 2023], v[v["연"] == 2024]
seen = a24["단지"].isin(a23["단지"]).to_numpy()      # 2023년에 거래가 있던 단지인가
deals = a23["단지"].value_counts()
print(len(a23), len(a24), a23["동이름"].nunique(), a23["단지"].nunique())
print(int((deals == 1).sum()), int(deals.median()), int(deals.max()))
print(int((~seen).sum()), round(float((~seen).mean() * 100), 1),
      int((~a24["동이름"].isin(a23["동이름"])).sum()))

def marks(actual, predicted, known):
    """오차율의 중앙값을 전체, 본 단지, 처음 보는 단지로 나눠 돌려준다."""
    return (off(actual, predicted), off(actual[known], predicted[known]),
            off(actual[~known], predicted[~known]))

lap = time.perf_counter()
ridge = fit(a23, "릿지")
took = time.perf_counter() - lap
base_test = {"릿지": marks(price(a24), ridge(a24), seen)}
print("릿지", *base_test["릿지"], f"{took:.1f}초")
lap = time.perf_counter()
boosted = fit(a23, "릿지+부스팅")
took = time.perf_counter() - lap
base_test["릿지+부스팅"] = marks(price(a24), boosted(a24), seen)
print("릿지+부스팅", *base_test["릿지+부스팅"], f"{took:.1f}초")

In [ ]:
names = ["가 단지", "나 단지", "다 단지"]
number = {name: k for k, name in enumerate(names)}  # 번호: 0, 1, 2
one_hot = torch.eye(3)                              # 원-핫: 행 하나가 단지 하나
torch.manual_seed(0)
W = torch.randn(3, 2)                               # 입력 3개, 유닛 2개인 층의 가중치
print(W)
print(one_hot[number["나 단지"]], one_hot[number["나 단지"]] @ W)
cells_needed = len(a23) * a23["단지"].nunique()     # 원-핫 행렬의 칸 수
print(len(a23), a23["단지"].nunique(), round(cells_needed * 4 / 1e6))

## 8.2 임베딩 층: 범주마다 숫자 여러 개를 학습시키기

In [ ]:
torch.manual_seed(0)
table = nn.Embedding(3, 2)                 # 범주 3개, 범주마다 숫자 2개
print(table.weight.shape)
print(table(torch.tensor([1, 1, 2])))      # 1번, 1번, 2번 행을 꺼낸다

same = one_hot[[1, 1, 2]] @ table.weight   # 원-핫에 같은 표를 가중치로 곱한다
print(torch.equal(same, table(torch.tensor([1, 1, 2]))))

loss = (table(torch.tensor([1, 1, 2])) ** 2).sum()
loss.backward()
print(table.weight.grad)                   # 꺼내지 않은 0번 행의 기울기는 0이다

In [ ]:
try:
    table(torch.tensor([3]))               # 표에 없는 번호
except IndexError as error:
    print("IndexError:", error)

In [ ]:
def encoder(made, month):
    """made의 동과 단지에 1부터 번호를 매긴다. 0번은 made에 없던 동과 단지가 쓴다.
    month: "그대로", "자르기"(made의 마지막 달보다 큰 달은 마지막 달로), "빼기"."""
    dongs = pd.Index(sorted(made["동이름"].unique()))
    complexes = pd.Index(sorted(made["단지"].unique()))
    last = made["달"].max()
    def numbers(df):
        when = np.minimum(df["달"], last) if month == "자르기" else df["달"]
        cols = [np.log(df["면적"]), df["층"], df["건축년도"], when]
        return np.column_stack(cols[:3] if month == "빼기" else cols).astype(np.float64)
    center, scale = numbers(made).mean(axis=0), numbers(made).std(axis=0)
    def encode(df):
        x = torch.tensor((numbers(df) - center) / scale, dtype=torch.float32)
        dong = torch.tensor(dongs.get_indexer(df["동이름"]) + 1)   # 없으면 -1 + 1 = 0
        complex_ = torch.tensor(complexes.get_indexer(df["단지"]) + 1)
        return x, dong, complex_
    return encode, len(dongs) + 1, len(complexes) + 1

encode, n_dong, n_complex = encoder(a23, "그대로")
x, dong, complex_ = encode(a24.head(4))
print(n_dong, n_complex)
print(x.shape, dong.tolist(), complex_.tolist())
print(int((encode(a24)[2] == 0).sum()))

## 8.3 임베딩을 넣은 신경망 학습시키기

In [ ]:
class TabNet(nn.Module):
    """숫자 열과 동 임베딩 8개, 단지 임베딩 16개를 이어 붙여 은닉층 2개에 넣는다."""
    def __init__(self, n_dong, n_complex, n_numbers, hidden=64):
        super().__init__()
        self.dong = nn.Embedding(n_dong, 8)
        self.complex = nn.Embedding(n_complex, 16)
        self.body = nn.Sequential(nn.Linear(n_numbers + 8 + 16, hidden), nn.ReLU(),
                                  nn.Linear(hidden, hidden), nn.ReLU(),
                                  nn.Linear(hidden, 1))

    def forward(self, x, dong, complex_):
        joined = torch.cat([x, self.dong(dong), self.complex(complex_)], dim=1)
        return self.body(joined)[:, 0]

net = TabNet(n_dong, n_complex, 4)
count = lambda module: sum(param.numel() for param in module.parameters())
print(count(net.dong), count(net.complex), count(net.body), count(net))
print(net(x, dong, complex_).shape)

In [ ]:
def train(made, target, month="그대로", drop=0.1, epochs=30, seed=0):
    """made로 학습시키고, 거래 표를 받아 target과 같은 단위로 예측하는 함수를 돌려준다.
    drop: 학습할 때 이 비율만큼의 거래에서 단지 번호를 0번으로 바꾼다."""
    encode, n_dong, n_complex = encoder(made, month)
    x, dong, complex_ = encode(made)
    mean, std = target.mean(), target.std()
    z = torch.tensor((target - mean) / std, dtype=torch.float32)
    torch.manual_seed(seed)
    net = TabNet(n_dong, n_complex, x.shape[1])
    optimizer = torch.optim.Adam(net.parameters(), lr=0.003)
    steps = epochs * -(-len(z) // 256)                       # 배치 크기 256
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, steps)
    shuffler = torch.Generator().manual_seed(seed)
    for epoch in range(epochs):
        net.train()
        order = torch.randperm(len(z), generator=shuffler)
        for start in range(0, len(z), 256):
            rows = order[start:start + 256]
            picked = complex_[rows]
            if drop > 0:
                hide = torch.rand(len(rows), generator=shuffler) < drop
                picked = picked.masked_fill(hide, 0)     # 처음 보는 단지로 바꾼다
            optimizer.zero_grad()
            nn.functional.mse_loss(net(x[rows], dong[rows], picked), z[rows]).backward()
            optimizer.step()
            scheduler.step()
    net.eval()
    def predict(df):
        with torch.no_grad():
            return net(*encode(df)).numpy().astype(np.float64) * std + mean
    predict.net = net
    return predict

lap = time.perf_counter()
model = train(a23, price(a23))
print(*marks(price(a24), model(a24), seen), f"{time.perf_counter() - lap:.1f}초")

In [ ]:
def trial(made, test, month="그대로", on_ridge=False, drop=0.1):
    """시드 0~4로 학습시켜 test의 예측(로그 가격) 5개를 돌려준다.
    on_ridge가 참이면 릿지 회귀의 잔차를 학습시키고 릿지의 예측에 더한다."""
    line = fit(made, "릿지") if on_ridge else (lambda df: 0.0)
    target = price(made) - line(made)
    return [train(made, target, month, drop, seed=seed)(test) + line(test)
            for seed in range(5)]

def pad(text, width):
    wide = sum(ord(ch) > 0x1100 for ch in text)
    return text + " " * max(width - len(text) - wide, 1)

def show(name, predictions, test, known):
    rows = np.array([marks(price(test), predicted, known) for predicted in predictions])
    parts = [f"{np.median(col):>5.2f} ({col.min():.2f}~{col.max():.2f})".ljust(22)
             for col in rows.T]
    print(pad(name, 26) + "".join(parts).rstrip())
    return rows

results = {}
HEAD = pad(" 전체", 22) + pad(" 본 단지", 22) + " 처음 보는 단지"
print(pad("2024년 오차율(%)", 26) + HEAD)
for drop in [0.0, 0.1, 0.3]:
    results["그대로", False, drop] = trial(a23, a24, drop=drop)
    show(f"단지를 가리는 비율 {drop}", results["그대로", False, drop], a24, seen)

In [ ]:
for drop in [0.0, 0.1]:
    net = train(a23, price(a23), drop=drop).net
    torch.manual_seed(0)
    start = TabNet(n_dong, n_complex, 4).complex.weight      # 같은 시드의 시작값
    moved = (net.complex.weight - start).norm(dim=1)
    print(drop, f"{moved[0].item():.3f}", f"{moved[1:].median().item():.3f}")

## 8.4 학습 범위 밖의 달을 넣을 때 생기는 일

In [ ]:
early, late = a23[a23["달"] <= 9], a23[a23["달"] >= 10]   # 1~9월로 학습, 10~12월로 검증
known = late["단지"].isin(early["단지"]).to_numpy()
print(len(early), len(late), round(float((~known).mean() * 100), 1))
base_valid = {kind: marks(price(late), fit(early, kind)(late), known)
              for kind in ("릿지", "릿지+부스팅")}
for kind, row in base_valid.items():
    print(kind, *row)

choices = [(month, on_ridge) for on_ridge in (False, True)
           for month in ("그대로", "자르기", "빼기")]
def label(month, on_ridge):
    return ("릿지+신경망" if on_ridge else "신경망") + ", 달 " + month

valid = {}
print(pad("검증(2023년 10~12월)", 26) + HEAD)
for month, on_ridge in choices:
    valid[month, on_ridge] = show(label(month, on_ridge),
                                  trial(early, late, month, on_ridge), late, known)

In [ ]:
test = {}
print(pad("2024년 오차율(%)", 26) + HEAD)
for month, on_ridge in choices:
    if (month, on_ridge, 0.1) not in results:
        results[month, on_ridge, 0.1] = trial(a23, a24, month, on_ridge)
    test[month, on_ridge] = show(label(month, on_ridge),
                                 results[month, on_ridge, 0.1], a24, seen)

In [ ]:
busy = deals.index[0]                              # 2023년에 거래가 가장 많은 단지
homes = a23[a23["단지"] == busy]
typical = homes[homes["면적"] == homes["면적"].mode()[0]]
one = typical.iloc[[0]].copy()
one["층"] = int(typical["층"].median())
print(busy, len(homes), float(one["면적"].iloc[0]), int(one["층"].iloc[0]),
      int(one["건축년도"].iloc[0]), len(typical))

months = np.arange(1, 25)
sheet = pd.concat([one] * 24, ignore_index=True)
sheet["달"] = months
paths = {"릿지": np.exp(ridge(sheet)), "릿지+부스팅": np.exp(boosted(sheet))}
for month in ["그대로", "자르기"]:
    paths["신경망, 달 " + month] = np.array(
        [np.exp(train(a23, price(a23), month, seed=seed)(sheet)) for seed in range(5)])
for name, path in paths.items():
    path = np.atleast_2d(path)
    middle = np.median(path, axis=0)
    span = lambda m: f"({path[:, m - 1].min():.2f}~{path[:, m - 1].max():.2f})"
    print(pad(name, 20) + f"{middle[0]:.2f}  {middle[11]:.2f} {span(12)}  "
          f"{middle[17]:.2f}  {middle[23]:.2f} {span(24)}")
actual = v[(v["단지"] == busy) & (v["면적"] == float(one["면적"].iloc[0]))]
traded = actual.groupby("달")["억"].median()
print(traded.reindex([1, 6, 12, 18, 24]).round(2).tolist(), len(actual))

## 8.5 그래디언트 부스팅과 다시 비교하기

In [ ]:
middle = v.groupby("달")["억"].median()
print(middle.round(2).tolist())
print(round(middle.loc[1:9].mean(), 2), round(middle.loc[10:12].mean(), 2),
      round(middle.loc[13:24].mean(), 2))

def error_by_month(predicted):
    gap = np.abs(np.exp(predicted - price(a24)) - 1) * 100
    return pd.Series(gap).groupby(a24["달"].to_numpy()).median()

picks = {"릿지+부스팅": boosted(a24),
         "신경망, 달 자르기": results["자르기", False, 0.1][0],
         "릿지+신경망, 달 자르기": results["자르기", True, 0.1][0]}
drift = {name: error_by_month(predicted) for name, predicted in picks.items()}
for name, series in drift.items():
    print(pad(name.split(",")[0], 13) + " ".join(f"{value:.2f}" for value in series))

In [ ]:
def explained(vectors, target):
    """vectors에 계수를 곱해 더한 식으로 target을 맞췄을 때의 결정계수."""
    A = np.column_stack([vectors, np.ones(len(vectors))])
    fitted = A @ np.linalg.lstsq(A, target, rcond=None)[0]
    return 1 - ((target - fitted) ** 2).sum() / ((target - target.mean()) ** 2).sum()

def level(key):
    """동이나 단지마다 2023년 ㎡당 가격의 중앙값에 로그를 취한 값."""
    return np.log((a23["억"] / a23["면적"]).groupby(a23[key]).median())

net = train(a23, price(a23), "자르기").net
torch.manual_seed(0)
fresh = TabNet(n_dong, n_complex, 4)                         # 학습시키기 전의 같은 모델
rows_of = lambda layer: layer.weight[1:].detach().numpy().astype(np.float64)
dong_level, complex_level = level("동이름"), level("단지")   # 이름순이 번호순이다
often = (deals.reindex(complex_level.index) >= 5).to_numpy()
print(len(dong_level), int(often.sum()))
for name, model in [("학습 전", fresh), ("학습 후", net)]:
    print(name, f"{explained(rows_of(model.dong), dong_level):.3f}",
          f"{explained(rows_of(model.complex)[often], complex_level[often]):.3f}")

vectors = rows_of(net.dong)
unit = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
per_area = np.exp(dong_level) * 1e4                          # ㎡당 만 원
for name in ["강남구 압구정동", "노원구 상계동", "마포구 아현동"]:
    k = dong_level.index.get_loc(name)
    near = np.argsort(-(unit @ unit[k]))[1:4]
    print(f"{name}({per_area.iloc[k]:,.0f}): " + ", ".join(
        f"{dong_level.index[j]}({per_area.iloc[j]:,.0f})" for j in near))

In [ ]:
import sys, platform, sklearn

print(sys.version.split()[0], np.__version__, torch.__version__, sklearn.__version__)
print(platform.machine(), torch.get_default_dtype(), torch.get_num_threads())

In [ ]:
!pip -q install koreanize-matplotlib

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib
from matplotlib.colors import to_rgba
from matplotlib.ticker import FuncFormatter

NAVY, BRICK, TEAL, GRAY = "#16324f", "#b5532a", "#2a7f73", "#9aa1ab"
BOX = {"fc": "white", "ec": "none", "alpha": 0.9, "pad": 1.5}


def frame(ax, title=None, color=NAVY, grid="y"):
    """그래프마다 반복하는 꾸밈: 제목, 보조선, 테두리 정리."""
    if title:
        ax.set_title(title, fontsize=9, loc="left", color=color, weight="bold")
    ax.grid(axis=grid, color="#e3e7ec", lw=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=9)
    ax.spines[["top", "right"]].set_visible(False)


def plain_ticks(axis):
    """로그 축의 눈금을 10의 거듭제곱 대신 보통 숫자로 적는다."""
    axis.set_major_formatter(FuncFormatter(lambda val, _: f"{val:,.0f}" if val >= 1 else f"{val:.7f}".rstrip("0")))

In [ ]:
# 그림 8-4
fig, ax = plt.subplots(figsize=(7, 5.2))
ax.axvspan(12.5, 24.6, color="#f1f2f4", lw=0)
ax.text(12.8, 24.6, "2024년 (학습에 쓰지 않은 달)", fontsize=9, color="#5b6470", va="top")
ax.plot(months, paths["릿지"], color=GRAY, lw=1.8, label="릿지 회귀")
ax.plot(months, paths["릿지+부스팅"], color=NAVY, lw=1.8, label="릿지 + 부스팅")
for k, row in enumerate(paths["신경망, 달 그대로"]):
    ax.plot(months, row, color=BRICK, lw=1.1, label="신경망, 달을 그대로 넣음 (시드 5개)" if k == 0 else None)
ax.plot(months, np.median(paths["신경망, 달 자르기"], axis=0), color=TEAL, lw=2.2, label="신경망, 12보다 큰 달을 12로 바꿈 (중앙값)")
ax.scatter(traded.index, traded.values, s=24, color="#3a4350", zorder=6, label="실제 거래의 달별 중앙값")
ax.set_xlim(0.4, 24.6); ax.set_ylim(16, 25); ax.set_xticks([1, 4, 7, 10, 13, 16, 19, 22, 24])
ax.set_xlabel("달 (1은 2023년 1월, 24는 2024년 12월)", fontsize=9); ax.set_ylabel("가격 (억 원)", fontsize=9)
ax.legend(frameon=False, fontsize=9, loc="upper left", bbox_to_anchor=(0.0, 0.93))
frame(ax)
fig.tight_layout()
fig.savefig("fig_8_4.png", dpi=300)
plt.show()

In [ ]:
# 그림 8-5
fig, ax = plt.subplots(figsize=(7, 4.4))
for lo, hi, col, name in [(0.5, 9.5, "#e9edf3", "학습 (1~9월)"), (9.5, 12.5, "#e3f0ec", "검증 (10~12월)"), (12.5, 24.5, "#f7ece6", "평가 (2024년)")]:
    ax.axvspan(lo, hi, color=col, lw=0)
    ax.text((lo + hi) / 2, 10.9, name, fontsize=9, color="#3a4350", ha="center", va="top")
ax.plot(middle.index, middle.values, color=NAVY, lw=1.8, marker="o", ms=4.5)
ax.set_xlim(0.5, 24.5); ax.set_ylim(8, 11); ax.set_xticks([1, 4, 7, 10, 13, 16, 19, 22, 24])
ax.set_xlabel("달 (1은 2023년 1월, 24는 2024년 12월)", fontsize=9); ax.set_ylabel("그 달 거래 가격의 중앙값 (억 원)", fontsize=9)
frame(ax)
fig.tight_layout()
fig.savefig("fig_8_5.png", dpi=300)
plt.show()

In [ ]:
# 그림 8-6
lines = [("릿지 회귀", base_valid["릿지"][0], base_test["릿지"][0], GRAY), ("릿지 + 부스팅", base_valid["릿지+부스팅"][0], base_test["릿지+부스팅"][0], NAVY)]
for month, on_ridge in choices:
    lines.append((label(month, on_ridge), np.median(valid[month, on_ridge][:, 0]), np.median(test[month, on_ridge][:, 0]), TEAL if month == "자르기" else BRICK))
lines = lines[::-1]
fig, ax = plt.subplots(figsize=(7, 5.0))
for k, (name, before, after, col) in enumerate(lines):
    shown = min(after, 8.6)
    ax.plot([before, shown], [k, k], color=col, lw=1.2)
    ax.scatter([before], [k], s=46, facecolor="white", edgecolor=col, lw=1.3, zorder=5)
    ax.scatter([shown], [k], s=46, color=col, zorder=5, marker="o" if after <= 8.6 else ">")
    ax.text(before, k + 0.28, f"{before:.2f}", fontsize=9, color=col, ha="center")
    ax.text(shown, k - 0.42, f"{after:.2f}", fontsize=9, color=col, ha="center", weight="bold")
ax.set_yticks(range(len(lines))); ax.set_yticklabels([row[0] for row in lines], fontsize=9)
ax.set_xlim(4, 8.9); ax.set_ylim(-0.8, len(lines) - 0.4)
ax.set_xlabel("오차율의 중앙값 (%). 빈 동그라미: 검증(2023년 10~12월), 찬 동그라미: 2024년", fontsize=9)
frame(ax, grid="x")
fig.tight_layout()
fig.savefig("fig_8_6.png", dpi=300)
plt.show()

In [ ]:
# 그림 8-7
fig, ax = plt.subplots(figsize=(7, 4.6))
for (name, series), col, marker in zip(drift.items(), [NAVY, BRICK, TEAL], ["o", "s", "D"]):
    ax.plot(series.index - 12, series.values, color=col, lw=1.8, marker=marker, ms=4.5, label=name)
    ax.text(0.85, series.values[0], f"{series.values[0]:.2f}", fontsize=9, color=col, ha="right", va="center", weight="bold")
    ax.text(12.15, series.values[-1], f"{series.values[-1]:.2f}", fontsize=9, color=col, ha="left", weight="bold",
            va={NAVY: "bottom", TEAL: "top"}.get(col, "center"))
ax.set_xlim(0.1, 12.9); ax.set_ylim(3, 7.5); ax.set_xticks(range(1, 13))
ax.set_xlabel("2024년의 달", fontsize=9); ax.set_ylabel("그 달 거래의 오차율 중앙값 (%)", fontsize=9)
ax.legend(frameon=False, fontsize=9, loc="upper left")
frame(ax)
fig.tight_layout()
fig.savefig("fig_8_7.png", dpi=300)
plt.show()